In [ ]:
# Cell 1: Install Required Packages
!pip install ASE
!pip install mace-torch ase rdkit weas-widget

In [ ]:
# Cell 2: Import Required Libraries
import numpy as np
import matplotlib.pyplot as plt
from ase import Atoms
from ase.build import bulk, molecule
from mace.calculators import mace_mp, mace_off

print("All imports successful.")

In [ ]:
# Cell 3: Load MACE-OFF

print("Loading MACE-OFF (medium model)...")
calc_mol = mace_off(model="medium", default_dtype="float64")
print("MACE-OFF loaded.")

In [ ]:
# Cell 4: Optimize Water Molecule (Gas)
from ase.optimize import BFGS

H_2_O = Atoms('HOH', positions=[(0,0,0), (0.757, 0.586, 0), (1.514, 0, 0)],
            cell=[15, 15, 15], pbc=False)
H_2_O.calc = calc_mol

opt = BFGS(H_2_O, logfile=None)
opt.run(fmax=0.001)

bond_length_OH = H_2_O.get_distance(0, 1)
bond_length_HH = H_2_O.get_distance(1, 2)

E_H_2_O = H_2_O.get_potential_energy()

print(f"Optimized OH bond length: {bond_length_OH:.4f} Å   (exp: 0.9572 Å)")
print(f"H₂O total energy:          {E_H_2_O:.6f} eV")

In [ ]:
# Cell 5: Calculate Water (gas) Properties
from ase.build import molecule
from ase.optimize import QuasiNewton
from ase.thermochemistry import IdealGasThermo
from ase.vibrations import Vibrations
from ase.units import kJ, mol

atoms_H2O = molecule('H2O')
atoms_H2O.calc = calc_mol
dyn = QuasiNewton(atoms_H2O, logfile=None)
dyn.run(fmax=0.01)
potentialenergy = atoms_H2O.get_potential_energy()

vib = Vibrations(atoms_H2O, name='h2o_vib')
vib.clean()
vib.run()
vib_energies = vib.get_energies()
vib_energies = np.array([e.real for e in vib_energies if e.real > 0.01])


thermo = IdealGasThermo(
    vib_energies=vib_energies,
    potentialenergy=potentialenergy,
    atoms=atoms_H2O,
    geometry='nonlinear', # Linear (Straight Line) or Nonlinear (Bent in any way)
    symmetrynumber=2, # How many times you can rotate the molecule and get the same configuration
    spin=0, # 0.5 for each unpaired electrons
)
#G_H2O = thermo.get_gibbs_energy(temperature=298.15, pressure=101325.0, verbose=False)
H_H2O = thermo.get_enthalpy(temperature=298.15, verbose=False)

H_H2O_kJ = H_H2O * (1/(kJ/mol))

print(f"Enthalpy of H₂O at 298 K: {H_H2O_kJ:.4f} kJ/mol")

In [ ]:
# Cell 6: Calculate Enthalpy for H2

atoms_H2 = molecule('H2')
atoms_H2.calc = calc_mol
dyn = QuasiNewton(atoms_H2, logfile=None)
dyn.run(fmax=0.01)
potentialenergy = atoms_H2.get_potential_energy()

vib = Vibrations(atoms_H2, name='h2_vib')
vib.clean()
vib.run()
vib_energies = vib.get_energies()
vib_energies = np.array([e.real for e in vib_energies if e.real > 0.01])


thermo = IdealGasThermo(
    vib_energies=vib_energies,
    potentialenergy=potentialenergy,
    atoms=atoms_H2,
    geometry='linear', # Linear (Straight Line) or Nonlinear (Bent in any way)
    symmetrynumber=2, # How many times you can rotate the molecule and get the same configuration
    spin=0, # 0.5 for each unpaired electrons
)
#G_H2 = thermo.get_gibbs_energy(temperature=298.15, pressure=101325.0, verbose=False)
H_H2 = thermo.get_enthalpy(temperature=298.15, verbose=False)

H_H2_kJ = H_H2 * (1/(kJ/mol))

print(f"H2 Enthalpy at 298 K: {H_H2_kJ:.4f} kJ/mol")

In [ ]:
# Cell 7: Calculate Enthalpy for O2

atoms_O2 = molecule('O2')
atoms_O2.calc = calc_mol
dyn = QuasiNewton(atoms_O2, logfile=None)
dyn.run(fmax=0.01)
potentialenergy = atoms_O2.get_potential_energy()

vib = Vibrations(atoms_O2, name='o2_vib')
vib.clean()
vib.run()
vib_energies = vib.get_energies()
vib_energies = np.array([e.real for e in vib_energies if e.real > 0.01])


thermo = IdealGasThermo(
    vib_energies=vib_energies,
    potentialenergy=potentialenergy,
    atoms=atoms_O2,
    geometry='linear', # Linear (Straight Line) or Nonlinear (Bent in any way)
    symmetrynumber=2, # How many times you can rotate the molecule and get the same configuration
    spin=1, # 0.5 for each unpaired electrons
)
#G_O2 = thermo.get_gibbs_energy(temperature=298.15, pressure=101325.0, verbose=False)
H_O2 = thermo.get_enthalpy(temperature=298.15, verbose=False)

H_O2_kJ = H_O2 * (1/(kJ/mol))

print(f"O2 Enthalpy at 298 K: {H_O2_kJ:.4f} kJ/mol")

In [ ]:
# Cell 8: Calculate Enthalpy Change for H2O (g) using H2 and O2 and Error

dH_Exp = H_H2O_kJ - (H_H2_kJ + 0.5 * H_O2_kJ)
dH_Act = -241.82

print(f"Enthalpy change for H2O formation at 298 K: {(dH_Exp):.4f} kJ/mol")
print(f"Experimental: -241.82 kJ/mol")

Percent_Error = abs((dH_Exp - dH_Act) / dH_Act) * 100
print(f"Percent Error: {Percent_Error:.2f}%")